# Lab 17 — Multimodal Alignment Map
**Goal:** Represent image and text tokens, fuse them, and reason about context and compute.

Free-first: CPU or free Colab where possible; expensive paths are clearly marked optional.



In [ ]:
import torch
image_tokens=1024
text_tokens=256
d=128
vision=torch.randn(1,image_tokens,d)
text=torch.randn(1,text_tokens,d)
print(vision.shape,text.shape)


In [ ]:
fused=torch.cat([vision,text],dim=1)
print("fused:",fused.shape)
print("sequence tokens:",fused.shape[1])


In [ ]:
# Compare three strategies at a high level.
strategies={
"concat": image_tokens+text_tokens,
"compress_4x": image_tokens//4+text_tokens,
"compress_16x": image_tokens//16+text_tokens
}
print(strategies)


### Answer
Multimodal systems are interface designs: representation, projection/alignment, fusion, training signals, and evaluation. Compressing visual tokens lowers context cost but can discard information; concatenation preserves more detail but raises compute.


## Primary source
https://cs336.stanford.edu/


## Deepening: visual tokens are part of context

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
text_tokens = [256, 1024, 4096]
visual_tokens = [64, 256, 1024]

for t in text_tokens:
    for v in visual_tokens:
        total = t + v
        print(f"text={t:4d} visual={v:4d} total_positions={total:5d} relative_L2={total*total/(256+64)**2:.2f}")


## Break experiment

Hold text constant and increase visual token count.

Predict how attention-related memory scales.

Then discuss the quality/resource trade-off of higher visual resolution or more visual tokens.

### Report

image representation → visual tokens → total context → memory consequence → quality hypothesis

Complete templates/EXPERIMENT_CARD.md for the final result.


## 7. Research exercise — multimodal data begins with modality distribution


In [ ]:
from datasets import load_dataset
from collections import Counter

# COCO-style image-text data is a practical bridge from text-only to multimodal records.
ds=load_dataset("HuggingFaceM4/COCO",split="train",streaming=True)
rows=[]
for row in ds:
    rows.append(row)
    if len(rows)>=100: break

print("rows:",len(rows))
print("fields:",list(rows[0].keys()) if rows else [])
for r in rows[:2]:
    print("keys:",list(r.keys()))
    print("caption sample:",str(r.get("sentences") or r.get("caption") or r)[:500])


<details><summary><strong>Expected findings / answer</strong></summary>

A multimodal training record has more structure than “text plus an image.” You need to know which fields contain the image, captions, IDs, and metadata, how many captions map to each image, and whether modalities are aligned. The first research artifact should therefore be a **schema and alignment audit** before model architecture changes. If this dataset/configuration changes upstream, inspect the schema rather than assuming these field names remain constant.
</details>
